# EEEM004 MSc Project — Audio Source Separation Pipeline
**Automated Soundtrack Personalisation for Neurodivergent Listeners Using Generative AI**

---

**Student:** Louis Ilett  
**Supervisor:** Prof Philip Jackson  
**Module:** EEEM004 — MSc Project, University of Surrey

---

## Pipeline Overview

This notebook implements an initial prototype of the soundtrack personalisation pipeline. The stages are:

1. **Audio acquisition** — download a chosen audio/video source via YouTube
2. **Source separation** — separate the mixture into stems using Demucs (htdemucs model)
3. **Speech enhancement** — apply noise reduction to the isolated vocals stem
4. **Speaker diarization** — identify and label individual speakers within the vocals stem
5. **Visualisation** — plot waveforms and spectrograms for analysis

> **Note:** Demucs separates into `vocals / drums / bass / other` rather than `dialogue / music / SFX`. This is a known limitation discussed in the project report. The vocals stem approximates dialogue, and the other stem approximates background audio.

---
## 1. Setup & Installation

Run this cell first. After it completes, **restart the runtime** (Runtime > Restart runtime) before continuing to avoid numpy version conflicts introduced by pyannote.

In [ ]:
!pip install demucs --quiet
!pip install librosa soundfile matplotlib --quiet
!pip install yt-dlp --quiet
!pip install noisereduce --quiet
!pip install pyannote.audio --quiet

print("All packages installed. Please restart the runtime before running the next cell.")

---
## 2. Imports

Run after restarting the runtime.

In [ ]:
import os
import torch
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import noisereduce as nr
from IPython.display import Audio, display
from pyannote.audio import Pipeline

print("Imports successful.")

---
## 3. Authentication

The speaker diarization model (pyannote) requires a Hugging Face token. 

**Do not paste your token directly into the code.** Instead:
1. Click the 🔑 **Secrets** tab in the left sidebar
2. Add a secret named `HF_TOKEN` with your token as the value
3. Run the cell below to load it securely

You also need to have accepted the terms at:
- huggingface.co/pyannote/speaker-diarization-3.1
- huggingface.co/pyannote/segmentation-3.0

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

if HF_TOKEN:
    print("Hugging Face token loaded successfully.")
else:
    print("WARNING: HF_TOKEN not found. Add it to Colab Secrets (key icon in left sidebar).")

---
## 4. Audio Acquisition

Paste any YouTube URL below. For best results, choose a clip with:
- Clear dialogue or speech
- Background music and/or sound effects
- No longer than a few minutes (separation is slow on long files)

Good choices: podcast clips, film/TV scenes, interviews with background music.

In [ ]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # <-- replace this

INPUT_PATH = "/content/test_audio.mp3"

print(f"Downloading from: {YOUTUBE_URL}")
!yt-dlp -x --audio-format mp3 -o "{INPUT_PATH}" "{YOUTUBE_URL}"

print("\nDownload complete. Playing original audio:")
display(Audio(INPUT_PATH))

---
## 5. Source Separation with Demucs

Runs the `htdemucs` model to separate the mixture into four stems:

| Stem | Project mapping |
|------|-----------------|
| `vocals` | Dialogue / speech |
| `other` | Background ambience / SFX |
| `drums` | Percussive elements (less relevant) |
| `bass` | Low-frequency elements (less relevant) |

This may take several minutes depending on the length of the audio.

In [ ]:
OUTPUT_DIR = "/content/demucs_output"

print("Running Demucs separation...")
!python -m demucs -n htdemucs "{INPUT_PATH}" -o "{OUTPUT_DIR}"
print("Separation complete.")

---
## 6. Listen to Separated Stems

Play back each stem and note your observations. Key things to listen for:
- How cleanly is speech isolated in the vocals stem?
- What bleeds into the wrong stem (artefacts)?
- What does the `other` stem contain?

In [ ]:
# Dynamically find the output folder — avoids hardcoding the filename
out_base = os.path.join(OUTPUT_DIR, "htdemucs")
folders = [f for f in os.listdir(out_base) if os.path.isdir(os.path.join(out_base, f))]
out_dir = os.path.join(out_base, folders[0])

print(f"Stems found in: {out_dir}\n")

stem_labels = {
    "vocals.wav": "Vocals (approx. dialogue)",
    "other.wav": "Other (approx. background / SFX)",
    "drums.wav": "Drums (percussive elements)",
    "bass.wav": "Bass (low-frequency elements)"
}

for filename, label in stem_labels.items():
    path = os.path.join(out_dir, filename)
    if os.path.isfile(path):
        print(f"--- {label} ---")
        display(Audio(path))

---
## 7. Speech Enhancement (Noise Reduction)

Applies noise reduction to the Demucs vocals stem to reduce bleed artefacts.

**Parameter:** `prop_decrease` controls aggressiveness (0.0 = no reduction, 1.0 = maximum).  
A value of `0.8` was found to work well — higher values introduce over-processing artefacts.

In [ ]:
VOCALS_PATH = os.path.join(out_dir, "vocals.wav")
ENHANCED_PATH = "/content/vocals_enhanced.wav"

# Load vocals stem
y, sr = librosa.load(VOCALS_PATH, sr=None, mono=True)

# Use first 0.5s as noise profile (assumes silence or near-silence at the start)
noise_sample = y[:int(sr * 0.5)]

# Apply noise reduction
print("Applying noise reduction...")
reduced = nr.reduce_noise(y=y, sr=sr, y_noise=noise_sample, prop_decrease=0.8)

# Save enhanced output
sf.write(ENHANCED_PATH, reduced, sr)
print("Enhancement complete.\n")

print("Before enhancement (raw Demucs vocals):")
display(Audio(VOCALS_PATH))

print("\nAfter enhancement:")
display(Audio(ENHANCED_PATH))

---
## 8. Speaker Diarization

Uses pyannote's pretrained speaker diarization model to identify who spoke when within the vocals stem. This enables individual speakers to be isolated — useful for scenarios where a neurodivergent listener may want to focus on a single speaker.

**Prerequisites:** HF_TOKEN loaded in Cell 3, and both pyannote model terms accepted on Hugging Face.

In [ ]:
print("Loading speaker diarization pipeline...")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)

print("Running diarization on vocals stem...")
diarization = diarization_pipeline(VOCALS_PATH)

print("\nSpeaker timeline:")
print(f"{'Speaker':<12} {'Start':>8} {'End':>8} {'Duration':>10}")
print("-" * 44)
for turn, _, speaker in diarization.itertracks(yield_label=True):
    duration = turn.end - turn.start
    print(f"{speaker:<12} {turn.start:>7.1f}s {turn.end:>7.1f}s {duration:>9.1f}s")

---
## 9. Extract Individual Speaker Audio

Uses the diarization output to extract each speaker's speech segments into a separate audio file.

In [ ]:
y_vocals, sr_vocals = librosa.load(VOCALS_PATH, sr=None, mono=True)

# Collect segments per speaker
speakers = {}
for turn, _, speaker in diarization.itertracks(yield_label=True):
    start_sample = int(turn.start * sr_vocals)
    end_sample = int(turn.end * sr_vocals)
    if speaker not in speakers:
        speakers[speaker] = []
    speakers[speaker].append(y_vocals[start_sample:end_sample])

# Concatenate and save each speaker
print(f"Found {len(speakers)} speaker(s).\n")
for speaker, segments in speakers.items():
    combined = np.concatenate(segments)
    out_path = f"/content/speaker_{speaker}.wav"
    sf.write(out_path, combined, sr_vocals)
    print(f"--- {speaker} ({len(segments)} segments, {len(combined)/sr_vocals:.1f}s total) ---")
    display(Audio(out_path))

---
## 10. Visualisation

Plots spectrograms of the original mixture, vocals stem, and enhanced vocals for comparison. Useful for analysing what separation has achieved spectrally.

In [ ]:
# Load all three signals for comparison
OTHER_PATH = os.path.join(out_dir, "other.wav")

signals = [
    (INPUT_PATH,    "Original Mixture"),
    (VOCALS_PATH,   "Demucs — Vocals Stem"),
    (ENHANCED_PATH, "Enhanced Vocals (after noise reduction)"),
    (OTHER_PATH,    "Demucs — Other Stem (background/SFX)"),
]

fig, axes = plt.subplots(len(signals), 1, figsize=(14, 4 * len(signals)))
fig.suptitle("Spectrogram Comparison — Source Separation Pipeline", fontsize=13, fontweight="bold", y=1.01)

for ax, (path, title) in zip(axes, signals):
    y_sig, sr_sig = librosa.load(path, sr=None, mono=True)
    # Trim to same length for fair comparison
    max_samples = min(len(y_sig), sr_sig * 60)  # max 60s
    y_sig = y_sig[:max_samples]

    D = librosa.amplitude_to_db(np.abs(librosa.stft(y_sig)), ref=np.max)
    img = librosa.display.specshow(D, sr=sr_sig, x_axis="time", y_axis="log", ax=ax)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Frequency (Hz)")
    fig.colorbar(img, ax=ax, format="%+2.0f dB")

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.savefig("/content/pipeline_spectrograms.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to /content/pipeline_spectrograms.png")

---
## 11. Observations Log

Fill in your observations after running the pipeline. This will be useful for your progress review write-up.

In [ ]:
observations = """
PIPELINE OBSERVATIONS
======================
Date:
Audio source (YouTube URL or description):

--- Source Separation (Demucs) ---
Vocals stem quality (subjective):
Other stem contents (what could you hear?):
Noticeable bleed artefacts (e.g. sounds in wrong stem):

--- Speech Enhancement (noisereduce) ---
Improvement over raw vocals stem? Y/N:
Any over-processing artefacts introduced?:
Best prop_decrease value found:

--- Speaker Diarization (pyannote) ---
Number of speakers detected:
Was the speaker timeline accurate? Y/N:
Any misattributed segments?:

--- General ---
Key limitations observed:
  1.
  2.
Questions to raise with supervisor:
  1.
  2.
"""

with open("/content/pipeline_observations.txt", "w") as f:
    f.write(observations)

print("Observation template saved to /content/pipeline_observations.txt")
print("Fill this in after listening — it will be useful for your progress review.")